# Stage 2 — Instruction Fine-Tuning / SFT (Healthcare FAQ Assistant)

**Goal:** teach the model to *answer healthcare questions* using the instruction dataset (`data/instruction_dataset.jsonl`, 100+ Q&A pairs).

Pipeline: Base → Stage 1: Non-Instruction FT → **[Stage 2: SFT]** → Stage 3: DPO

You can start from the Stage-1 model (`RESUME_FROM_STAGE1 = True`) or from the base model.

> ⚠️ Educational project — general health information only, not medical advice.

## 0. Install dependencies (Colab)

In [1]:
# Run once on a fresh Colab GPU runtime (Runtime -> Change runtime type -> T4 GPU).
# Unsloth installs compatible transformers / peft / trl / bitsandbytes itself --
# do NOT pin an old trl (e.g. trl<0.12): it passes `tokenizer=` to Trainer.__init__(),
# which newer transformers removed, causing:
#   TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'
%%capture
!pip install -q unsloth unsloth_zoo
# After installing, restart the runtime once (Runtime -> Restart session) before continuing.


In [2]:
import unsloth  # Important: import Unsloth early
import torch
import time
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

assert torch.cuda.is_available(), "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4


## 0b. Colab bootstrap — get repo files & set REPO_DIR

In [3]:
# ============================================================
#  COLAB BOOTSTRAP  --  make repo files available + set REPO_DIR
#  Pick ONE method by setting BOOTSTRAP below.
# ============================================================
import os

BOOTSTRAP = "drive"   # "drive" (recommended) | "clone" | "local"

if BOOTSTRAP == "drive":
    # 1) Copy the `healthcare-ai-assistant-finetuning` folder into your Google Drive.
    # 2) Adjust the path below if you placed it somewhere other than MyDrive root.
    #    Drive is recommended because outputs/ persist across sessions, so Stage 1->2->3 chain.
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["REPO_DIR"] = "/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning"

elif BOOTSTRAP == "clone":
    # Ephemeral: /content is wiped on disconnect. Run all stages in one session,
    # or set PUSH_TO_HUB=True so each stage's model is saved to the Hugging Face Hub.
    REPO_URL = "https://github.com/your-username/healthcare-faq-assistant.git"
    DEST = "/content/healthcare-faq-assistant"
    if not os.path.isdir(DEST):
        os.system(f"git clone {REPO_URL} {DEST}")
    os.environ["REPO_DIR"] = DEST

else:  # "local" -- running outside Colab, from inside the notebooks/ folder
    os.environ["REPO_DIR"] = ".."

print("REPO_DIR =", os.environ.get("REPO_DIR"))
assert os.path.isdir(os.path.join(os.environ["REPO_DIR"], "data")), \
    "REPO_DIR is wrong: no data/ folder found. Fix the path in this cell."

Mounted at /content/drive
REPO_DIR = /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning


## 1. Select base model

In [4]:
# ============================================================
#  MODEL SELECTION  --  change MODEL_NAME to switch base model
# ============================================================
MODEL_OPTIONS = {
    "qwen2.5-0.5b":   "unsloth/Qwen2.5-0.5B",
    "llama-3.2-1b":   "unsloth/Llama-3.2-1B",
    "qwen2.5-1.5b":   "unsloth/Qwen2.5-1.5B",
    "tinyllama-1.1b": "unsloth/tinyllama",
    "gemma-2-2b":     "unsloth/gemma-2-2b",
}

MODEL_NAME = "qwen2.5-0.5b"   # <-- change this one line to pick a model
MODEL_REPO = MODEL_OPTIONS[MODEL_NAME]
print(f"Selected model: {MODEL_NAME}  ->  {MODEL_REPO}")

Selected model: qwen2.5-0.5b  ->  unsloth/Qwen2.5-0.5B


## 2. Paths & system prompt

In [5]:
import os

# If you cloned the repo in Colab, point REPO_DIR at the repo root.
# This notebook lives in <repo>/notebooks/, so the repo root is one level up.
REPO_DIR   = os.environ.get("REPO_DIR", "..")
DATA_DIR   = os.path.join(REPO_DIR, "data")
OUTPUT_DIR = os.path.join(REPO_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Data dir:  ", os.path.abspath(DATA_DIR))
print("Output dir:", os.path.abspath(OUTPUT_DIR))

Data dir:   /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/data
Output dir: /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs


In [6]:
# Shared system prompt used for instruction formatting / inference
SYSTEM_PROMPT = (
    "You are a Healthcare FAQ Assistant. You provide clear, general health information for "
    "educational purposes only. You are not a substitute for professional medical advice, "
    "diagnosis, or treatment. Always recommend consulting a qualified healthcare professional, "
    "and advise seeking emergency care for urgent symptoms."
)

## 2b. Hugging Face Hub config (optional)

In [ ]:
# ============================================================
#  HUGGING FACE HUB  --  set these to push your trained models
# ============================================================
PUSH_TO_HUB  = False                 # set True to upload after training
HF_USERNAME  = "mannyiyer"     # <-- your Hugging Face username
HF_TOKEN     = ""                     # <-- a WRITE token from https://huggingface.co/settings/tokens

# In Colab you can store the token as a secret instead of pasting it:
#   from google.colab import userdata
#   HF_TOKEN = userdata.get('HF_TOKEN')

if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    print("Logged in to Hugging Face Hub as", HF_USERNAME)
else:
    print("PUSH_TO_HUB disabled (or no token). Models will be saved locally only.")

## 3. Load the model

If `RESUME_FROM_STAGE1` is True and `outputs/stage1_merged` exists, we continue from the non-instruction-tuned model; otherwise we start from the base model.

In [7]:
from unsloth import FastLanguageModel
import torch, os

max_seq_length = 2048
RESUME_FROM_STAGE1 = True   # set False to start from the base model

STAGE1_MERGED = os.path.join(OUTPUT_DIR, "stage1_merged")
if RESUME_FROM_STAGE1 and os.path.isdir(STAGE1_MERGED):
    load_from = STAGE1_MERGED
    print("Continuing from Stage 1:", load_from)
else:
    load_from = MODEL_REPO
    print("Starting from base model:", load_from)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = load_from,
    max_seq_length = max_seq_length,
    dtype          = None,
    load_in_4bit   = True,
)

Continuing from Stage 1: /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_merged
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_merged does not have a padding token! Will use pad_token = <<|PAD_TOKEN|>>.


In [8]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

# ---- 1. Load an untouched copy of the base model ----
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_REPO,       # raw base, not stage1_merged
    max_seq_length = max_seq_length,
    dtype          = None,
    load_in_4bit   = True,
)

# ---- 2. Make sure BOTH tokenizers have a chat template ----
# Qwen2.5 base checkpoints ship without one; apply the same ChatML
# fallback used during SFT training so formats match.
if base_tokenizer.chat_template is None:
    base_tokenizer = get_chat_template(base_tokenizer, chat_template="chatml")

if tokenizer.chat_template is None:   # the SFT tokenizer from this session
    tokenizer = get_chat_template(tokenizer, chat_template="chatml")

FastLanguageModel.for_inference(base_model)

# ---- 3. Your questions ----
my_questions = [
    "How can I apply for sick leave when I have the flu?",
    "What temperature is considered a fever in adults?",
    "Should I take antibiotics for a common cold?",
    "Can I stop my blood pressure medication if I feel fine?",
    "What should I do if my 2-month-old baby has a fever?",
    "How do I recognize the signs of a stroke?",
    "How much paracetamol can I take if my headache won't go away?",
    "What's a good way to lose weight quickly?",
    "How can I manage my type 2 diabetes?",
    "What can this assistant help me with?",
]

gen_kwargs = dict(
    max_new_tokens     = 200,
    max_length         = None, # Explicitly setting max_length to None to prevent the warning
    do_sample          = True,
    temperature        = 0.7,
    top_p              = 0.9,
    repetition_penalty = 1.15,
)

def ask_model(mdl, tok, question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    inputs = tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        out = mdl.generate(input_ids=inputs, **gen_kwargs)
    # Decode only the newly generated tokens
    return tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

# ---- 4. Compare base vs SFT ----
for q in my_questions:
    print("QUESTION :", q)
    print("BASE     :", ask_model(base_model, base_tokenizer, q))
    print("SFT      :", ask_model(model, tokenizer, q))
    print("=" * 80)

# Optional: free GPU memory afterwards
del base_model, base_tokenizer
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/521M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

...

unsloth/qwen2.5-0.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


QUESTION : How can I apply for sick leave when I have the flu?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

BASE     : Hello! As an AI system, I'm here to help you with any questions related to your daily life and well-being. When it comes to applying for sick leave due to illness such as the flu, there are several steps you should follow:

1. **Check Eligibility**: First, check if you're eligible for sick leave based on your employer's policies. Some employers may offer flexible scheduling options that allow employees to take time off without needing approval from their supervisors.

2. **Self-Report**: If you feel ill but don't want to go through formal documentation because of privacy concerns, consider self-reporting your condition at home using apps like HealthAppSolutions' "Self-Report" tool provided by HealthApps Solutions LLC (HAPS). This will give you peace of mind knowing your status is confidentially reported internally within HAPS rather than externally.

3. **Contact Your Employer**: Reach out directly to your employer regarding eligibility. They might be able to assist you in f

## 4. Set the chat template

We use the model's built-in chat template when available. For models without one (e.g. TinyLlama) we fall back to a simple ChatML template via Unsloth.

In [9]:
from unsloth.chat_templates import get_chat_template

if tokenizer.chat_template is None:
    tokenizer = get_chat_template(tokenizer, chat_template="chatml")
    print("Applied fallback ChatML template.")
else:
    print("Using the models built-in chat template.")

Using the models built-in chat template.


## 5. Load & format the instruction dataset

In [10]:
import json, os
from datasets import Dataset

path = os.path.join(DATA_DIR, "instruction_dataset.jsonl")
rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
print(f"Loaded {len(rows)} instruction examples")

def to_text(ex):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": ex["instruction"]},
        {"role": "assistant", "content": ex["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = Dataset.from_list([to_text(r) for r in rows])
print("\nExample formatted record:\n")
print(dataset[0]["text"][:600])

Loaded 112 instruction examples

Example formatted record:

<|im_start|>system
You are a Healthcare FAQ Assistant. You provide clear, general health information for educational purposes only. You are not a substitute for professional medical advice, diagnosis, or treatment. Always recommend consulting a qualified healthcare professional, and advise seeking emergency care for urgent symptoms.<|im_end|>
<|im_start|>user
What temperature is considered a fever in adults?<|im_end|>
<|im_start|>assistant
For most adults, a body temperature at or above 38 degrees Celsius (100.4 degrees Fahrenheit) is considered a fever. Mild fevers are often a normal sign tha


## 6. Apply LoRA (QLoRA adapters)

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.6.9 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


## 7. Train (SFT)

In [12]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# Note: no manual DataCollatorForLanguageModeling needed -- SFTTrainer builds its
# own causal-LM collator. The old explicit collator + `tokenizer=` argument were
# workarounds for the trl/transformers version mismatch, now fixed in the install cell.

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,      # new API: was `tokenizer=` in older trl
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = os.path.join(OUTPUT_DIR, "stage2_logs"),
        report_to = "none",
    ),
)
trainer_stats = trainer.train()
trainer_stats


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/112 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 112 | Num Epochs = 3 | Total steps = 42
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.205499
2,3.270820
3,3.134473
4,3.125237
5,2.780364
6,2.460006
7,2.156831
8,1.936620
9,1.719978
10,1.611232


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_logs/checkpoint-42/tokenizer_config.json.


TrainOutput(global_step=42, training_loss=1.3745722884223575, metrics={'train_runtime': 108.9277, 'train_samples_per_second': 3.085, 'train_steps_per_second': 0.386, 'total_flos': 99859475255040.0, 'train_loss': 1.3745722884223575, 'epoch': 3.0})

## 8. Save the SFT adapter + merged model (input to Stage 3)

In [13]:
STAGE2_ADAPTER = os.path.join(OUTPUT_DIR, "stage2_sft")
STAGE2_MERGED  = os.path.join(OUTPUT_DIR, "stage2_merged")

model.save_pretrained(STAGE2_ADAPTER)
tokenizer.save_pretrained(STAGE2_ADAPTER)
print("Saved SFT adapter ->", STAGE2_ADAPTER)

model.save_pretrained_merged(STAGE2_MERGED, tokenizer, save_method="merged_16bit")
print("Saved merged SFT model ->", STAGE2_MERGED)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_sft/tokenizer_config.json.


Saved SFT adapter -> /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_sft
Detected local model directory: /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage1_merged
Copied tokenizer.model from local model directory


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:41<00:00, 41.55s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_merged`
Saved merged SFT model -> /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_merged


### Optional — push the Stage model to the Hugging Face Hub

Runs only if `PUSH_TO_HUB = True` and a write token is set above. Pushes both the LoRA adapter (small) and the merged 16-bit model.

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    repo_adapter = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage2-sft"
    repo_merged  = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage2-sft-merged"

    # LoRA adapter
    model.push_to_hub(repo_adapter, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_adapter, token=HF_TOKEN)
    print("Pushed adapter ->", repo_adapter)

    # Merged 16-bit model (ready for inference / vLLM / TGI)
    model.push_to_hub_merged(repo_merged, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    print("Pushed merged model ->", repo_merged)
else:
    print("Skipped Hub upload (PUSH_TO_HUB is False or no token).")

### Reload this model later from the Hugging Face Hub

Once pushed, you (or anyone) can load it directly — no retraining needed:

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage2-sft-merged",
    max_seq_length = 2048, load_in_4bit = True,
)
```

Or with plain Transformers (CPU or GPU):

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
repo = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage2-sft-merged"
tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo)
```

## 9. Inference after SFT

In [15]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, use_cache=True,
                         do_sample=False, temperature=0.0)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return text.strip()

for q in [
    "How can I apply for sick leave when I have the flu?",
    "What temperature is considered a fever in adults?",
    "Should I take antibiotics for a common cold?",
    "Can I stop my blood pressure medication if I feel fine?",
    "What should I do if my 2-month-old baby has a fever?",
    "How do I recognize the signs of a stroke?",
    "How much paracetamol can I take if my headache won't go away?",
    "What's a good way to lose weight quickly?",
    "How can I manage my type 2 diabetes?",
    "What can this assistant help me with?",
]:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 80)

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I apply for sick leave when I have the flu?


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: You can apply for sick leave by submitting a written application and providing your medical history. You may be eligible for up to five days of leave, but you must complete the application within 10 days of your diagnosis. You can also request a leave of absence for a specific reason, such as caring for a family member or attending a medical emergency.
--------------------------------------------------------------------------------
Q: What temperature is considered a fever in adults?


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

A: You should only take paracetamol for a short period of time, and you should not exceed the maximum daily dose. If your headache is severe, persistent, or has no apparent cause, you should seek urgent medical care.
--------------------------------------------------------------------------------
Q: What's a good way to lose weight quickly?


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Losing weight quickly can be difficult, so it's best to start slowly and gradually reduce calories. Aim for at least 150 minutes of moderate aerobic activity per week, along with muscle-strengthening exercises. A diet low in added sugars, salt, and saturated fats can also help. If you are struggling, talk to a healthcare professional.
--------------------------------------------------------------------------------
Q: How can I manage my type 2 diabetes?


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Managing type 2 diabetes involves maintaining a healthy diet, regular exercise, and regular monitoring of blood sugar. A healthcare professional can help you develop a personalized plan. Regular check-ups are important, and if you have difficulty managing your diabetes, speak with a clinician.
--------------------------------------------------------------------------------
Q: What can this assistant help me with?
A: I can provide general health information for educational purposes only. If you have a specific health question or condition, please ask a healthcare professional who can provide a personalized advice.
--------------------------------------------------------------------------------


## Done — Stage 2 complete ✅

Next: open **`dpo_alignment.ipynb`** — it loads `outputs/stage2_merged` as the SFT model and aligns it with the preference dataset.